# Learning across problems: structured, then drifting problems

Do the mice learn to learn? Samborska et al. (2022, *Nat Neurosci*, Fig. 1) show three signs in a
two-port reversal task: the logistic-regression weights on past choices and outcomes grow across
problems (Fig. 1i-l), the mice switch to the new good port faster after a reversal late in
training (Fig. 1f), and they make fewer out-of-sequence pokes in later problems (Fig. 1g,h). Here
each analysis follows the mice through their structured (blocked) problems and then their drifting
ones:

1. **Choice-logit weights by problem** (`src/behavior_analysis/choice_logit/logit_by_problem.py`),
   fitted to whole problems and to the early and late trials of each problem.
2. **Choice probabilities around block changes** by training stage
   (`src/behavior_analysis/reversals/reversal_curves_by_stage.py`), the analogue of Fig. 1f.
3. **Where the mice poke as they meet a new layout** by training stage
   (`src/behavior_analysis/pokes/port_finding_by_stage.py`), the analogue of Fig. 1h.

The training stages (`src/behavior_analysis/data/training_stages.py`) are the first two structured
problems, the first two drifting problems and each mouse's last two drifting problems.


In [ ]:
# ruff: noqa: E402  (the repository root is put on sys.path first)
import sys
from pathlib import Path

from IPython.display import Image, display

REPO_ROOT = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "src/behavior_import").is_dir()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.behavior_visualization.choice_logit import plot_logit_by_problem as logit_plots
from src.behavior_visualization.pokes import plot_port_finding_by_stage as poke_plots
from src.behavior_visualization.reversals import plot_reversal_curves_by_stage as reversal_plots
from src.pipelines.choice_logit import logit_by_problem as logit_pipe
from src.pipelines.common.cohort_analysis import load_cohort
from src.pipelines.common.training_stages import staged_problems
from src.pipelines.pokes.port_finding_by_stage import run_port_finding_by_stage
from src.pipelines.reversals.reversal_curves_by_stage import run_reversal_curves_by_stage


In [ ]:
# ---- parameters (edit here) ------------------------------------------------------------
COHORTS = [1, 3]
DATA_ROOT = None  # None: BANDIT_DATA_ROOT. Loading is cache-first and never writes to the share.
# The last drifting problem kept per cohort (None: all); as the rank-signature notebook.
MAX_DRIFTING_BY_COHORT = {1: 7, 3: None}
OUTPUT_DIR = REPO_ROOT / "results" / "figures" / "learning-across-problems"


## Load

Every cohort is imported once (from the newest import cache). Structured problems are the blocked
problems before a mouse's first drifting problem; blocked problems that come later (cohort 1, after
surgery) and drifting problems after the cutoff are left out.


In [ ]:
imported = {c: load_cohort(c, DATA_ROOT, use_cache=True, from_cache=True) for c in COHORTS}
trials_by_cohort = {c: i.trials_by_problem for c, i in imported.items()}
port_finding_sessions = {c: i.log.port_finding_sessions if i.log else {} for c, i in imported.items()}
staged = staged_problems(trials_by_cohort, MAX_DRIFTING_BY_COHORT)
display(staged.groupby("stage").agg(mice=("mouse", "nunique"), problems=("mouse", "size")))


## 1. Choice-logit weights by problem

The three-arm choice logit is fitted to each mouse's problems one at a time, in two codings of the
history:

- **Magnitude** (the standard logit): per port, choice (1 if chosen) and reward (the pulses received
  there). The reward weight is the analogue of the paper's outcome x choice term, per pulse; the
  choice weight is the pull of a past choice that paid 0.
- **Outcome x choice** (the paper's coding): choice, and outcome x choice (+1 if the choice paid
  anything, -1 if it paid 0). The choice weight is the average pull of a past choice, rewarded or
  not; the outcome x choice weight is half the difference between rewarded and unrewarded
  (win-stay versus lose-shift). The magnitude is ignored, as the paper's reward was binary.

Each coding is also fitted to each problem's first 150 trials and to the rest. Learning to learn
predicts that the early-trial weights grow across problems and approach the late-trial weights.
The caption file beside each figure has the slope and step tests.


In [ ]:
trials = logit_pipe.load_training_trial_table(COHORTS, trials_by_cohort=trials_by_cohort)
logit = logit_pipe.run_logit_by_problem(trials, MAX_DRIFTING_BY_COHORT, OUTPUT_DIR)
for coding, result in logit.items():
    for base in logit_plots.FIGURE_NAMES:
        name = logit_plots.figure_name(base, coding)
        display(Image(filename=str(OUTPUT_DIR / f"{name}.png")))
    display(result.tests.round(4))
    display(result.early_late_tests.round(4))


## 2. Choice probabilities around block changes (Fig. 1f)

P(new best), P(previous best) and P(third) around the good block changes whose best arm changed,
per training stage, and each mouse's mean P(new best) over trials 1-10 after the change.


In [ ]:
reversals = run_reversal_curves_by_stage(trials_by_cohort, staged, OUTPUT_DIR)
display(Image(filename=str(OUTPUT_DIR / f"{reversal_plots.REVERSAL_CURVES_BY_STAGE}.png")))
display(reversals.tests.round(4))


## 3. Where the mice poke as they meet a new layout (Fig. 1h)

The fraction of pokes at the current choice ports, the previous problem's ports, the initiation
port and other ports, in port finding, the first 50 task trials and the first 3 blocks, per
training stage. In sequence is the next step of the trial: a current choice port after initiation,
the initiation port after the choice (the paper's out-of-sequence pokes are choice-port visits after
a choice).


In [ ]:
ports = run_port_finding_by_stage(trials_by_cohort, port_finding_sessions, staged, OUTPUT_DIR)
display(Image(filename=str(OUTPUT_DIR / f"{poke_plots.PORT_FINDING_BY_STAGE}.png")))
display(ports.tests.round(4))
